# Preprocessing

In [10]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from src import config

print("✅ Setup complete.")

✅ Setup complete.


In [11]:
# Generic loading mechanism
INTERIM_DATA_PATH = '../data/interim/water_quality_mvp_baseline.parquet' 
df = pd.read_parquet(INTERIM_DATA_PATH)

print(f"Loaded shape: {df.shape}")
display(df.head(3))

Loaded shape: (9319, 10)


,Latitude,Longitude,Sample Date,swir22,NDMI,MNDWI,pet,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,2011-01-02,7645.0,0.185538,0.195595,174.2,128.912,555.0,10.0
1,-26.861111,28.884722,2011-01-03,10574.0,0.124566,-0.180134,124.1,74.720,162.9,163.0
2,-26.450000,28.085833,2011-01-03,14201.0,-0.083293,-0.252805,127.5,89.254,573.0,80.0


### Anchor Drop

In [12]:
META_COLS = ['Latitude', 'Longitude', 'Sample Date']
df = df.drop(columns=META_COLS)

print(f"Shape after dropping spatial anchors: {df.shape}")
print(f"Universal columns remaining: {df.columns.tolist()}")

Shape after dropping spatial anchors: (9319, 7)
Universal columns remaining: ['swir22', 'NDMI', 'MNDWI', 'pet', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']


### Train/Test Split

In [13]:
TARGET_COL = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

X = df.drop(columns=TARGET_COL)
y = df[TARGET_COL]

print(f"Features: {X.columns.tolist()}")
print(f"Targets:  {y.columns.tolist()}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

Features: ['swir22', 'NDMI', 'MNDWI', 'pet']
Targets:  ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
X shape: (9319, 4), y shape: (9319, 3)


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print(f"Training: X={X_train.shape}, y={y_train.shape}")
print(f"Testing:  X={X_test.shape}, y={y_test.shape}")

Training: X=(7455, 4), y=(7455, 3)
Testing:  X=(1864, 4), y=(1864, 3)


### Feature Grouping

In [15]:
# All remaining features are strictly numeric and universal
num_cols = X_train.columns.tolist()

print(f"Universal features ({len(num_cols)}): {num_cols}")

Universal features (4): ['swir22', 'NDMI', 'MNDWI', 'pet']


### Building Preprocessor

In [16]:
# --- Numeric Pipeline ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=False)),
    ('scaler', StandardScaler()),
])

# --- Combine ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
    ],
    remainder='drop',
)

preprocessor

,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### Fit and Transform

In [17]:
# Fit on training data ONLY, then transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Resulting feature count: {len(feature_names)}")
print(f"Feature names: {feature_names.tolist()}")

Resulting feature count: 4
Feature names: ['num__swir22', 'num__NDMI', 'num__MNDWI', 'num__pet']


### Save

In [18]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Rebuild DataFrames with proper column names
train_fe = pd.DataFrame(X_train_processed, columns=feature_names)
for col in TARGET_COL:
    train_fe[col] = y_train[col].values

test_fe = pd.DataFrame(X_test_processed, columns=feature_names)
for col in TARGET_COL:
    test_fe[col] = y_test[col].values

# Save as Parquet (preserves dtypes, faster, smaller than CSV)
train_fe.to_parquet('../data/processed/train_fe.parquet', index=False)
test_fe.to_parquet('../data/processed/test_fe.parquet', index=False)

# Save the fitted preprocessor pipeline
PIPELINE_PATH = '../models/preprocessor.joblib'
joblib.dump(preprocessor, PIPELINE_PATH)

print(f"Train set: {train_fe.shape}")
print(f"Test set:  {test_fe.shape}")
print(f"Pipeline saved to: {PIPELINE_PATH}")
print("Preprocessing complete.")

Train set: (7455, 7)
Test set:  (1864, 7)
Pipeline saved to: ../models/preprocessor.joblib
Preprocessing complete.
